# **California DMV Form Collection Pipeline**
This notebook colelcts publicly available California DMV forms.

## Import Required Libraries
Import the libraries used for web requests, HTML parsing, file management, and metadata generation.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
import os
import time

## Configure Web Requests
These settings create a reusable request function that identifies our crawler with a browser style user agent and a short delay to avoid overwhelming the DMV site.

In [ ]:
HEADERS = {
    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/137.0 Safari/537.36"
}

def polite_get(url):
    time.sleep(1)
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response

In [ ]:
def download_pdf(url, folder):

    os.makedirs(folder, exist_ok=True)

    filename = url.split("/")[-1].split("?")[0]

    path = os.path.join(folder, filename)

    response = polite_get(url)

    with open(path, "wb") as f:
        f.write(response.content)

    return path

## Explore the DMV Forms website
Before building the crawler, this section inspects the DMV Forms page to better understand its structure, available links, and how PDF documents are organized. This helped guide the implementation of the crawler later.

In [ ]:
DMV_URL = "https://www.dmv.ca.gov/portal/forms/"

response = polite_get(DMV_URL)

print(response.status_code)
print(response.url)
print(response.text[:500])

200
https://www.dmv.ca.gov/portal/forms/
<!DOCTYPE html>
<html lang="en-US" class="no-js" dir="ltr">
	<head>
		<meta charset="UTF-8" />
		<meta name="viewport" content="width=device-width, initial-scale=1" />
		<script>(function(html){html.className = html.className.replace(/\bno-js\b/,'js')})(document.documentElement);</script>
<meta name='robots' content='index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1' />
	<script>
		var dataLayer = dataLayer || [];
		dataLayer.push({"task":"N\/A","funcType":"resource po


In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "lxml")

links = soup.find_all("a", href=True)

print("Total links:", len(links))

for link in links[:30]:
    print(link["href"])

Total links: 79
#main
https://www.ca.gov/
/portal/
#js-site-header__main-nav
/portal/driver-licenses-identification-cards/real-id/
/portal/dmv-online/
#
/portal/mydmv
https://dmv.ca.gov/portal/vehicle-registration/
/portal/vehicle-registration/vehicle-registration-renewal/
/portal/vehicle-registration/new-registration/
/portal/vehicle-registration/online-replacement-sticker-or-registration-card/
/portal/vehicle-registration/insurance-requirements/suspended-vehicle-registration/
/portal/customer-service/records-request/online-vehicle-record-request/
/portal/vehicle-registration/license-plates-decals-and-placards/
https://dmv.ca.gov/portal/driver-licenses-identification-cards/
/portal/driver-licenses-identification-cards/driver-license-id-card-online-renewal/
/portal/driver-licenses-identification-cards/dl-id-online-app-edl-44/
/portal/driver-licenses-identification-cards/replace-your-driver-license-or-identification-dl-id-card/online-duplicate-driver-license-request/
/portal/california-

In [ ]:
sample_url = "https://www.dmv.ca.gov/portal/form/application-for-disabled-person-placard-or-plates-reg-195/"

response = polite_get(sample_url)

print(response.status_code)

soup = BeautifulSoup(response.text, "lxml")

pdfs = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if ".pdf" in href.lower():
        pdfs.append(href)

print(f"Found {len(pdfs)} PDFs")

for pdf in pdfs:
    print(pdf)

200
Found 0 PDFs


In [ ]:
links = soup.find_all("a", href=True)

print("Total links:", len(links))

for a in links:
    print(a["href"])

Total links: 0


In [ ]:
for a in soup.find_all("a", href=True):
    text = a.get_text(strip=True)
    if text:
        print(f"{text[:60]:60} -> {a['href']}")

In [ ]:
dmv_forms = []

for a in links:
    href = a["href"]

    if "/portal/form/" in href:
        dmv_forms.append(
            "https://www.dmv.ca.gov" + href
        )

print("Total DMV forms found:", len(dmv_forms))

for form in dmv_forms[:10]:
    print(form)

Total DMV forms found: 0


In [ ]:
DMV_FORMS_URL = "https://www.dmv.ca.gov/portal/forms/"

dmv_response = polite_get(DMV_FORMS_URL)

print(dmv_response.status_code)
print(dmv_response.headers.get("Content-Type"))

200
text/html; charset=UTF-8


In [ ]:
dmv_soup = BeautifulSoup(dmv_response.text, "lxml")

dmv_links = dmv_soup.find_all("a", href=True)

dmv_forms = []

for a in dmv_links:
    href = a["href"]

    if "/portal/form/" in href:
        full_url = urljoin("https://www.dmv.ca.gov", href)
        dmv_forms.append(full_url)

# remove duplicates
dmv_forms = list(dict.fromkeys(dmv_forms))

print("Total DMV forms found:", len(dmv_forms))

for form in dmv_forms[:20]:
    print(form)

Total DMV forms found: 16
https://www.dmv.ca.gov/portal/form/application-for-disabled-person-placard-or-plates-reg-195/
https://www.dmv.ca.gov/portal/form/application-for-replacement-or-transfer-of-title-reg-227/
https://www.dmv.ca.gov/portal/form/application-for-replacement-plates-stickers-documents-reg-156/
https://www.dmv.ca.gov/portal/form/application-for-title-or-registration-verification-of-vehicle-reg-343/
https://www.dmv.ca.gov/portal/form/bill-of-sale-reg-135-pdf/
https://www.dmv.ca.gov/portal/form/statement-of-facts-reg-256/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-english-adm-140/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-armenian-adm-140/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-chinese-adm-140-ch/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-farsi-adm-140-fa/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-punjabi-adm-14/
https://www.dmv.ca.gov/portal/form/langu

## Build the Form Discovery Process
After understanding the website, these helper functions were made to find DMV form pages throughout the website.

In [ ]:
from urllib.parse import urljoin

BASE_DMV = "https://www.dmv.ca.gov"

def find_dmv_forms(page_url):
    response = polite_get(page_url)
    soup = BeautifulSoup(response.text, "lxml")

    forms = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "/portal/form/" in href:
            forms.append(urljoin(BASE_DMV, href))

    return forms


all_dmv_forms = []

for page in range(1, 32):

    if page == 1:
        url = "https://www.dmv.ca.gov/portal/forms/"
    else:
        url = f"https://www.dmv.ca.gov/portal/forms/page/{page}/"

    forms = find_dmv_forms(url)

    print(f"Page {page}: {len(forms)} forms")

    all_dmv_forms.extend(forms)


# remove duplicates
all_dmv_forms = list(dict.fromkeys(all_dmv_forms))


print("\nTotal unique DMV forms:", len(all_dmv_forms))

for form in all_dmv_forms[:20]:
    print(form)

Page 1: 16 forms
Page 2: 16 forms
Page 3: 16 forms
Page 4: 16 forms
Page 5: 16 forms
Page 6: 15 forms
Page 7: 16 forms
Page 8: 16 forms
Page 9: 16 forms
Page 10: 16 forms
Page 11: 16 forms
Page 12: 16 forms
Page 13: 16 forms
Page 14: 16 forms
Page 15: 16 forms
Page 16: 16 forms
Page 17: 16 forms
Page 18: 16 forms
Page 19: 16 forms
Page 20: 16 forms
Page 21: 16 forms
Page 22: 16 forms
Page 23: 16 forms
Page 24: 16 forms
Page 25: 15 forms
Page 26: 16 forms
Page 27: 16 forms
Page 28: 16 forms
Page 29: 16 forms
Page 30: 16 forms
Page 31: 12 forms

Total unique DMV forms: 305
https://www.dmv.ca.gov/portal/form/application-for-disabled-person-placard-or-plates-reg-195/
https://www.dmv.ca.gov/portal/form/application-for-replacement-or-transfer-of-title-reg-227/
https://www.dmv.ca.gov/portal/form/application-for-replacement-plates-stickers-documents-reg-156/
https://www.dmv.ca.gov/portal/form/application-for-title-or-registration-verification-of-vehicle-reg-343/
https://www.dmv.ca.gov/portal/f

In [ ]:
for url in all_dmv_forms[:5]:
    response = polite_get(url)
    print(url, response.headers["Content-Type"])

https://www.dmv.ca.gov/portal/form/application-for-disabled-person-placard-or-plates-reg-195/ application/pdf
https://www.dmv.ca.gov/portal/form/application-for-replacement-or-transfer-of-title-reg-227/ application/pdf
https://www.dmv.ca.gov/portal/form/application-for-replacement-plates-stickers-documents-reg-156/ application/pdf
https://www.dmv.ca.gov/portal/form/application-for-title-or-registration-verification-of-vehicle-reg-343/ application/pdf
https://www.dmv.ca.gov/portal/form/bill-of-sale-reg-135-pdf/ application/pdf


## Crawl DMV Form Pages

Next the crawler iterates through every page of the DMV Forms directory, collects the links to forms, and removes duplicates.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


BASE_URL = "https://www.dmv.ca.gov"
FORMS_URL = "https://www.dmv.ca.gov/portal/forms/"


def get_dmv_form_links():

    all_forms = set()

    page = 1

    while True:

        if page == 1:
            url = FORMS_URL
        else:
            url = f"{FORMS_URL}page/{page}/"

        response = requests.get(url)

        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        page_forms = []

        for a in soup.find_all("a", href=True):

            href = a["href"]

            full_url = urljoin(BASE_URL, href)

            if "/portal/form/" in full_url:
                page_forms.append(full_url)


        page_forms = list(set(page_forms))


        if len(page_forms) == 0:
            break


        print(f"Page {page}: {len(page_forms)} forms")


        all_forms.update(page_forms)

        page += 1


    return sorted(all_forms)

In [ ]:
# there are clearly enough form s will cut short later
#dmv_forms = get_dmv_form_links()

#print("\nTotal:", len(dmv_forms))

Page 1: 16 forms
Page 2: 16 forms
Page 3: 16 forms
Page 4: 16 forms
Page 5: 16 forms
Page 6: 15 forms
Page 7: 16 forms
Page 8: 16 forms
Page 9: 16 forms
Page 10: 16 forms
Page 11: 16 forms
Page 12: 16 forms
Page 13: 16 forms
Page 14: 16 forms
Page 15: 16 forms
Page 16: 16 forms
Page 17: 16 forms
Page 18: 16 forms
Page 19: 16 forms
Page 20: 16 forms
Page 21: 16 forms
Page 22: 16 forms
Page 23: 15 forms
Page 24: 14 forms
Page 25: 14 forms
Page 26: 15 forms
Page 27: 16 forms
Page 28: 16 forms
Page 29: 16 forms
Page 30: 16 forms
Page 31: 12 forms
Page 32: 6 forms
Page 33: 6 forms
Page 34: 6 forms
Page 35: 6 forms
Page 36: 6 forms
Page 37: 6 forms
Page 38: 6 forms
Page 39: 6 forms
Page 40: 6 forms
Page 41: 6 forms
Page 42: 6 forms
Page 43: 6 forms
Page 44: 6 forms
Page 45: 6 forms
Page 46: 6 forms
Page 47: 6 forms
Page 48: 6 forms
Page 49: 6 forms
Page 50: 6 forms
Page 51: 6 forms
Page 52: 6 forms
Page 53: 6 forms
Page 54: 6 forms
Page 55: 6 forms
Page 56: 6 forms
Page 57: 6 forms
Page 58: 

KeyboardInterrupt: 

In [ ]:
for form in dmv_forms[:10]:
    print(form)

https://www.dmv.ca.gov/portal/form/language-access-complaint-form-tagalog-adm-140/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-spanish-adm-140sp/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-punjabi-adm-14/
https://www.dmv.ca.gov/portal/form/bill-of-sale-reg-135-pdf/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-armenian-adm-140/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-farsi-adm-140-fa/
https://www.dmv.ca.gov/portal/form/application-for-replacement-plates-stickers-documents-reg-156/
https://www.dmv.ca.gov/portal/form/application-for-replacement-or-transfer-of-title-reg-227/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-russian-adm-140-ru/
https://www.dmv.ca.gov/portal/form/language-access-complaint-form-vietnamese-adm-140/


In [ ]:
def get_dmv_form_links():

    all_forms = set()

    for page in range(1, 32):

        if page == 1:
            url = FORMS_URL
        else:
            url = f"{FORMS_URL}page/{page}/"

        response = requests.get(url)

        soup = BeautifulSoup(response.text, "html.parser")

        page_forms = []

        for a in soup.find_all("a", href=True):

            href = a["href"]

            full_url = urljoin(BASE_URL, href)

            if "/portal/form/" in full_url:
                page_forms.append(full_url)

        page_forms = list(set(page_forms))

        print(f"Page {page}: {len(page_forms)} forms")

        all_forms.update(page_forms)

    return sorted(all_forms)

In [ ]:
dmv_forms = get_dmv_form_links()

print("Total:", len(dmv_forms))

Page 1: 16 forms
Page 2: 16 forms
Page 3: 16 forms
Page 4: 16 forms
Page 5: 16 forms
Page 6: 15 forms
Page 7: 16 forms
Page 8: 16 forms
Page 9: 16 forms
Page 10: 16 forms
Page 11: 16 forms
Page 12: 16 forms
Page 13: 16 forms
Page 14: 16 forms
Page 15: 16 forms
Page 16: 16 forms
Page 17: 16 forms
Page 18: 16 forms
Page 19: 16 forms
Page 20: 16 forms
Page 21: 16 forms
Page 22: 16 forms
Page 23: 15 forms
Page 24: 14 forms
Page 25: 14 forms
Page 26: 15 forms
Page 27: 16 forms
Page 28: 16 forms
Page 29: 16 forms
Page 30: 16 forms
Page 31: 12 forms
Total: 305


## Identify Downloadable PDF Documents
I found that not every discovered page is a downloadable form. So this checks each page to see wether it has a PDF so that only valid files are included in the dataset

In [ ]:
def check_pdf_links(form_links):

    pdf_links = []

    for url in form_links:

        response = requests.get(url)

        content_type = response.headers.get("Content-Type","")

        if "application/pdf" in content_type:
            pdf_links.append(url)

    return pdf_links


dmv_pdf_links = check_pdf_links(dmv_forms)

print("PDFs:", len(dmv_pdf_links))

PDFs: 304


In [ ]:
for url in dmv_pdf_links[:20]:
    print(url)

https://www.dmv.ca.gov/portal/form/10-year-history-record-check-dl-939/
https://www.dmv.ca.gov/portal/form/10-year-history-record-check-spanish-dl-939-sp/
https://www.dmv.ca.gov/portal/form/287903/
https://www.dmv.ca.gov/portal/form/31099/
https://www.dmv.ca.gov/portal/form/34114/
https://www.dmv.ca.gov/portal/form/41965/
https://www.dmv.ca.gov/portal/form/50000-bond-exemption-application-ol-56/
https://www.dmv.ca.gov/portal/form/91495/
https://www.dmv.ca.gov/portal/form/abandoned-vehicle-application-for-salvage-certificate-or-nonrepairable-vehicle-certificate-reg-479/
https://www.dmv.ca.gov/portal/form/affidavit-for-transfer-without-probate-california-titled-vehicle-or-vessels-only-reg-5/
https://www.dmv.ca.gov/portal/form/affidavit-of-non-use-reg-5090/
https://www.dmv.ca.gov/portal/form/agreement-to-prepare-and-maintain-records-in-accordance-with-international-registration-plan-and-california-apportionment-requirements-mc-522-i/
https://www.dmv.ca.gov/portal/form/annual-report-of-aut

In [ ]:
from collections import Counter

extensions = Counter()

for url in dmv_pdf_links:
    filename = url.split("/")[-1]
    extensions[filename.split("-")[0]] += 1

extensions.most_common(20)

[('', 304)]

## Filter the Dataset
The translated form versions are removed so that we collect usable data.

In [ ]:
exclude_words = [
    "spanish",
    "chinese",
    "armenian",
    "farsi",
    "punjabi",
    "russian",
    "tagalog",
    "vietnamese",
    "korean"
]


filtered_dmv_links = [
    url for url in dmv_pdf_links
    if not any(word in url.lower() for word in exclude_words)
]


print("Before:", len(dmv_pdf_links))
print("After:", len(filtered_dmv_links))

Before: 304
After: 278


## Download DMV Documents

In [ ]:
import os
import requests
from pathlib import Path


# Create output directory
dmv_dir = Path("dmv_pdfs")
dmv_dir.mkdir(exist_ok=True)


downloaded_files = []
failed_downloads = []


for i, url in enumerate(filtered_dmv_links, start=1):

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Get filename from URL slug
        filename = url.rstrip("/").split("/")[-1] + ".pdf"

        file_path = dmv_dir / filename

        with open(file_path, "wb") as f:
            f.write(response.content)

        downloaded_files.append(str(file_path))

        print(f"{i}/{len(filtered_dmv_links)} downloaded: {filename}")

    except Exception as e:
        failed_downloads.append((url, str(e)))
        print(f"FAILED: {url}")
        print(e)


print("\nDownload complete")
print("Downloaded:", len(downloaded_files))
print("Failed:", len(failed_downloads))

1/278 downloaded: 10-year-history-record-check-dl-939.pdf
2/278 downloaded: 287903.pdf
3/278 downloaded: 31099.pdf
4/278 downloaded: 34114.pdf
5/278 downloaded: 41965.pdf
6/278 downloaded: 50000-bond-exemption-application-ol-56.pdf
7/278 downloaded: 91495.pdf
8/278 downloaded: abandoned-vehicle-application-for-salvage-certificate-or-nonrepairable-vehicle-certificate-reg-479.pdf
9/278 downloaded: affidavit-for-transfer-without-probate-california-titled-vehicle-or-vessels-only-reg-5.pdf
10/278 downloaded: affidavit-of-non-use-reg-5090.pdf
11/278 downloaded: agreement-to-prepare-and-maintain-records-in-accordance-with-international-registration-plan-and-california-apportionment-requirements-mc-522-i.pdf
12/278 downloaded: annual-report-of-autonomous-vehicle-disengagement-ol-311r.pdf
13/278 downloaded: application-for-approval-of-mature-driver-improvement-course-ol-1002.pdf
14/278 downloaded: application-for-authorization-for-lien-sale-of-a-lien-sale-for-a-vehicle-valued-over-4000-reg-656.

In [ ]:
list(dmv_dir.iterdir())[:10]

[PosixPath('dmv_pdfs/statement-of-multiple-county-use-of-vehicle-reg-6004.pdf'),
 PosixPath('dmv_pdfs/traffic-violator-school-foreign-language-approval-request.pdf'),
 PosixPath('dmv_pdfs/motorized-bicycle-instructions-application-reg-230.pdf'),
 PosixPath('dmv_pdfs/request-for-vehicle-vessel-automated-record-information-inf-1123.pdf'),
 PosixPath('dmv_pdfs/renewal-list.pdf'),
 PosixPath('dmv_pdfs/traffic-violator-school-branch-business-office-classroom-application-ol-712-pdf.pdf'),
 PosixPath('dmv_pdfs/notification-of-class-schedule-ol-854e.pdf'),
 PosixPath('dmv_pdfs/notice-of-change-of-address.pdf'),
 PosixPath('dmv_pdfs/salvage-vehicle-notice-of-retention-by-owner-reg-481.pdf'),
 PosixPath('dmv_pdfs/notice-of-intent-to-dispose-of-a-vehicle-valued-500-or-less-removed-by-a-public-agency-for-reasons-other-than-abandonment-reg-684.pdf')]

In [ ]:
from pathlib import Path
!pip install PyMuPDF
import fitz  # PyMuPDF


sample = list(dmv_dir.glob("*.pdf"))[0]

doc = fitz.open(sample)

text = ""

for page in doc:
    text += page.get_text()

print(sample.name)
print(text[:500])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 54.9 MB/s eta 0:00:00
statement-of-multiple-county-use-of-vehicle-reg-6004.pdf
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
    
 
 
STATEMENT OF MULTIPLE COUNTY USE OF VEHICLE 
(CVC §4004.5) 
MAIL TO DEPARTMENT OF MOTOR VEHICLES 
P.O. BOX 942869
 SACRAMENTO, CA 94269-0001 
YOU MAY PROVIDE THE FOLLOWING INFORMATION IF YOU RESIDE IN MORE THAN ONE COUNTY FOR 
A PERIOD OF MORE THAN 30 DAYS, OR USE YOUR VEHICLE IN A COUNTY OTHER THAN YOUR COUNTY 
OF LEGAL RESIDENCE FOR BUSINESS PURPOSES. ADDITIONAL PAGES MAY BE ATTACHED IF THE 
INFORMATION TO BE SUBMITTED EXCEED


## Generate Document Metadata
Metadata is created for each PDF, like file name, agency, form identifier, doc type, and file location. This is helpful when I later combine the data.

In [ ]:
import os
import pandas as pd
from pathlib import Path


def build_metadata(folder, agency):
    records = []

    for file in Path(folder).glob("*.pdf"):
        records.append({
            "filename": file.name,
            "filepath": str(file),
            "agency": agency
        })

    return records


uscis_records = build_metadata(
    "uscis_pdfs",
    "USCIS"
)

dmv_records = build_metadata(
    "dmv_pdfs",
    "DMV"
)


documents = pd.DataFrame(
    uscis_records + dmv_records
)


documents.head()

,filename,filepath,agency
0,statement-of-multiple-county-use-of-vehicle-re...,dmv_pdfs/statement-of-multiple-county-use-of-v...,DMV
1,traffic-violator-school-foreign-language-appro...,dmv_pdfs/traffic-violator-school-foreign-langu...,DMV
2,motorized-bicycle-instructions-application-reg...,dmv_pdfs/motorized-bicycle-instructions-applic...,DMV
3,request-for-vehicle-vessel-automated-record-in...,dmv_pdfs/request-for-vehicle-vessel-automated-...,DMV
4,renewal-list.pdf,dmv_pdfs/renewal-list.pdf,DMV


In [ ]:
documents["agency"].value_counts()

,count
agency,
DMV,277


In [ ]:
from pathlib import Path

print("DMV PDFs:", len(list(Path("dmv_pdfs").glob("*.pdf"))))

DMV PDFs: 277


## Export and Save Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/newstart_ai")

PROJECT_DIR.mkdir(exist_ok=True)

(PROJECT_DIR / "data/raw").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from pathlib import Path
import re


# Folder where DMV PDFs are stored
dmv_folder = Path("/content/dmv_pdfs")


def extract_dmv_form_number(filename):
    """
    Extract DMV form numbers from filenames.

    Examples:
    reg-227.pdf -> REG-227
    dl-939.pdf -> DL-939
    """

    match = re.search(
        r"(reg|dl|ol|adm|inf|mc)-?\d+",
        filename.lower()
    )

    if match:
        return match.group(0).upper()

    return None



dmv_metadata = []


for pdf in dmv_folder.glob("*.pdf"):

    filename = pdf.name

    dmv_metadata.append({
        "filename": filename,
        "agency": "DMV",
        "form_number": extract_dmv_form_number(filename),
        "document_type": "form",
        "filepath": str(pdf)
    })


dmv_df = pd.DataFrame(dmv_metadata)


# Save CSV
dmv_df.to_csv(
    "dmv_metadata.csv",
    index=False
)


print("DMV metadata saved!")
print("Rows:", len(dmv_df))

dmv_df.head()

DMV metadata saved!
Rows: 277


,filename,agency,form_number,document_type,filepath
0,statement-of-multiple-county-use-of-vehicle-re...,DMV,REG-6004,form,/content/dmv_pdfs/statement-of-multiple-county...
1,traffic-violator-school-foreign-language-appro...,DMV,None,form,/content/dmv_pdfs/traffic-violator-school-fore...
2,motorized-bicycle-instructions-application-reg...,DMV,REG-230,form,/content/dmv_pdfs/motorized-bicycle-instructio...
3,request-for-vehicle-vessel-automated-record-in...,DMV,INF-1123,form,/content/dmv_pdfs/request-for-vehicle-vessel-a...
4,renewal-list.pdf,DMV,None,form,/content/dmv_pdfs/renewal-list.pdf


In [ ]:
import pandas as pd
from pathlib import Path


dmv_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/dmv")

metadata = []

for pdf in dmv_folder.glob("*.pdf"):

    metadata.append({
        "filename": pdf.name,
        "filepath": str(pdf),
        "agency": "DMV",
        "document_type": "form"
    })


dmv_df = pd.DataFrame(metadata)


output_path = Path(
    "/content/drive/MyDrive/newstart_ai/data/metadata/dmv_metadata.csv"
)

dmv_df.to_csv(output_path, index=False)


print("DMV metadata saved!")
print("Rows:", len(dmv_df))

dmv_df.head()

DMV metadata saved!
Rows: 277


,filename,filepath,agency,document_type
0,statement-of-multiple-county-use-of-vehicle-re...,/content/drive/MyDrive/newstart_ai/data/raw/dm...,DMV,form
1,traffic-violator-school-foreign-language-appro...,/content/drive/MyDrive/newstart_ai/data/raw/dm...,DMV,form
2,motorized-bicycle-instructions-application-reg...,/content/drive/MyDrive/newstart_ai/data/raw/dm...,DMV,form
3,request-for-vehicle-vessel-automated-record-in...,/content/drive/MyDrive/newstart_ai/data/raw/dm...,DMV,form
4,renewal-list.pdf,/content/drive/MyDrive/newstart_ai/data/raw/dm...,DMV,form
